# Roblox Applied ML Interview — Guided Titanic Project
### From a raw CSV to a defensible classification model

**Audience:** You know ML concepts but need to rebuild coding fluency in **Pandas, NumPy, and scikit-learn**.  
**Interview target:** a **45-minute applied ML coding exercise** with data inspection, preprocessing, a working model, evaluation, and explanation.  
**Your goal here:** understand a **reusable process** — not memorize Titanic-specific facts.

**How to use this notebook:**
1. Run the **worked example** code cells in order. All of them form a working end-to-end pipeline.
2. At each **✍️ Practice** section, **write code into the TODO cell** before reading the answer.
3. Stuck? Open **Hint** first. If needed, expand **Reveal reference answer**. These are intentionally collapsed.
4. Exercises are **optional**: an untouched notebook can still run top to bottom without errors.
5. After Titanic, attempt the **45-minute transfer challenge** at the end without looking back.

**Estimated time:** 2–4 hours guided, then 45 minutes for the transfer mock. This is a learning workbook, **not** a script to memorize line-by-line.

## 0. The generic applied-ML workflow (works beyond Titanic)

| Interview stage | Question you are answering | Typical tools | Approx. time in a 45-min interview |
|---|---|---|---|
| **Define the task** | What is a row? What is the target? Is it regression or classification? | column names, problem statement | 0–5 min |
| **Inspect the data** | What is missing? What is the target balance? Any suspicious columns? | `head`, `info`, `isna`, `value_counts` | 5–15 min |
| **Split + prepare** | How do I prevent leakage while converting data into model inputs? | `train_test_split`, `ColumnTransformer`, `Pipeline` | 15–25 min |
| **Baseline + train** | Can I beat a simple guess? | `DummyClassifier`, logistic regression, random forest | 25–35 min |
| **Evaluate + explain** | How does it fail? Is the metric appropriate? What would I improve? | confusion matrix, F1, accuracy, cross-validation | 35–45 min |

**Transfer principle:** On a different dataset, your *questions and sequencing* should be similar even when the columns and prediction target change.

**Titanic task:** each row is a passenger; predict `Survived` (0 or 1) from information that plausibly existed before the outcome. This is **binary classification**. For something like predicting house prices, the same workflow applies but you would use a regression model and error metrics such as MAE / RMSE.

## 1. Environment and dataset

The notebook you uploaded was written for Kaggle and uses paths like `/kaggle/input/titanic/train.csv`. **This rewrite runs locally** as long as you either:

- Place the official Kaggle Titanic `train.csv` next to this notebook (or inside `data/`), **or**
- Have an internet connection the first time you run it. The notebook downloads the equivalent public 891-row training CSV and caches it in `data/titanic_train.csv`.

Install packages into your virtual environment:

**macOS/Linux:**
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m jupyter lab
```

**Windows PowerShell:**
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m jupyter lab
```

For use in VS Code, open the notebook and select the `.venv` Python kernel. Notebook dependencies are listed in the accompanying `requirements.txt` file. If using Jupyter outside of that ZIP, `pip install pandas numpy matplotlib scikit-learn jupyterlab ipykernel` works.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__, "| NumPy:", np.__version__)

### Load the CSV — why this step matters

A CSV is a file of rows and columns; **Pandas `read_csv()` turns it into a DataFrame**. In an interview, the dataset may already be in memory as `df` — you would start at the inspection step instead.

**Do not use** `/kaggle/input/` as a mandatory local path. We search a few convenient local locations first. If nothing is there, we download and cache the training CSV. If you are offline, the error message tells you what to do.

In [ ]:
possible_paths = [
    Path("data/titanic_train.csv"),
    Path("data/train.csv"),
    Path("train.csv"),
    Path("titanic.csv"),
    Path("/kaggle/input/titanic/train.csv"),
]
found = next((p for p in possible_paths if p.is_file()), None)

if found:
    df = pd.read_csv(found)
    print(f"Loaded local CSV: {found}")
else:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    try:
        df = pd.read_csv(url)
    except Exception as exc:
        raise RuntimeError(
            "Could not download the Titanic CSV. Download the official Titanic "
            "train.csv from https://www.kaggle.com/competitions/titanic/data "
            "and place it next to this notebook or inside data/."
        ) from exc
    df.to_csv(DATA_DIR / "titanic_train.csv", index=False)
    print("Downloaded public Titanic training CSV and cached it at data/titanic_train.csv")

required = {"Survived", "Sex", "Pclass", "Age", "Fare", "SibSp", "Parch", "Embarked"}
assert required.issubset(df.columns), "Wrong CSV or missing required columns: " + str(required - set(df.columns))
assert df["Survived"].dropna().isin([0, 1]).all(), "Survived should contain binary labels"
print("Data shape:", df.shape)
display(df.head(5))

### ✍️ Practice 1: Load and inspect the dataset

**Why this matters:** Every new dataset begins with knowing its dimensions and columns.

**Task:** Create `my_first_three` with the first three rows of `df`, and `number_of_columns` with the number of columns.

<details><summary>Hint (open only if stuck)</summary><p>Try .head(3) and .shape[1].</p></details>

In [ ]:
# TODO: write two lines below.
# my_first_three = ...
# number_of_columns = ...

In [ ]:
if "my_first_three" in globals() and "number_of_columns" in globals():
    assert my_first_three.shape[0] == 3
    assert number_of_columns == df.shape[1]
    print("Practice 1: passed")
else:
    print("Practice 1: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 1</b></summary>
<pre><code>my_first_three = df.head(3)
number_of_columns = df.shape[1]
print(number_of_columns)
display(my_first_three)</code></pre>
</details>

## 2. Understand each column, its type, and the target

**Question:** What do we have, and what are we trying to predict?

- `Survived`: target, a binary label; **never put this column in `X`**.
- `Pclass`: ticket class (1/2/3): numeric-looking but represents a category.
- `Sex`, `Embarked`: categorical features.
- `Age`, `Fare`: numeric features with missing or extreme values.
- `SibSp`, `Parch`: counts of family relations aboard.
- `PassengerId`, `Name`, `Ticket`, `Cabin`: initially exclude to keep a small, explainable baseline; they are **not automatically useless**. `Cabin` also has substantial missingness.

**Generic principle:** Distinguish *feature meaning* from *storage type* (`int` does not always mean continuous quantity).

In [ ]:
print("Rows, columns:", df.shape)
print("Column names:", df.columns.tolist())
print("\nTypes and non-null counts:")
df.info()
print("\nNumeric summaries:")
display(df.describe().T)
print("\nFirst three rows, selected columns:")
display(df.loc[:, ["Pclass", "Sex", "Age", "Fare", "Survived"]].head(3))

### Pandas in context: rows, columns, and filtering

- `df['Age']` → **Series** (one labeled column).
- `df[['Age', 'Fare']]` → **DataFrame** (two columns).
- `df.loc[condition, ['Age', 'Fare']]` → rows matching a **question**, specific columns.
- `df.iloc[0:5]` → first five rows by **position**, independent of index labels.

**Example question:** Which passengers were children under 12, and how were they distributed across classes? This tells us whether age may interact with ticket class; it is a hypothesis, not yet a modeling conclusion.

In [ ]:
children = df.loc[df["Age"] < 12, ["Pclass", "Sex", "Age", "Survived"]]
print("Passengers with recorded age under 12:", len(children))
display(children.head(8))

### ✍️ Practice 2: Filter for a meaningful subgroup

**Why this matters:** Filtering helps you investigate hypotheses and data quality before modeling.

**Task:** Build `third_class_adults` containing only rows where `Pclass == 3` AND `Age >= 18`; display the first five.

<details><summary>Hint (open only if stuck)</summary><p>Use df.loc[(condition1) &amp; (condition2), :]. Parenthesize BOTH conditions.</p></details>

In [ ]:
# TODO: select passengers in class 3 aged 18 or older.
# third_class_adults = ...
# display(third_class_adults.head())

In [ ]:
if "third_class_adults" in globals():
    assert (third_class_adults["Pclass"] == 3).all()
    assert (third_class_adults["Age"] >= 18).all()
    print("Practice 2: passed")
else:
    print("Practice 2: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 2</b></summary>
<pre><code>third_class_adults = df.loc[(df[&quot;Pclass&quot;] == 3) &amp; (df[&quot;Age&quot;] &gt;= 18), :]
display(third_class_adults.head())</code></pre>
</details>

## 3. Data-quality audit: what is missing?

**Why we do this:** Many ML models cannot accept missing numeric values; missingness can also tell us how the data was collected. We must choose a repair strategy **after** seeing the problem, not fill everything with 0 blindly.

**Generic process:** for each column inspect the **number missing**, **percentage missing**, and meaning of missingness. A sparse free-text column might be excluded; a mostly complete numeric column might be imputed.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(1),
    "dtype": df.dtypes.astype(str),
}).sort_values("missing_pct", ascending=False)
display(missing)

fig, ax = plt.subplots(figsize=(8, 3))
missing.loc[missing["missing_count"] > 0, "missing_pct"].plot.bar(ax=ax)
ax.set(ylabel="Missing values (%)", title="Where is the data incomplete?")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Why **not** fill every missing age with 0?

A missing `Age` is **unknown**, not a newborn. Filling it with zero creates a false cluster of infants. Instead, we'll estimate a representative value using **median age in the training split**. This is intentionally performed **inside** a scikit-learn pipeline so validation data cannot influence the imputer.

**Data leakage:** any information you would not legitimately know at prediction time (including statistics learned from validation/test rows) can make scores artificially optimistic.

### ✍️ Practice 3: Missing-data inspection

**Why this matters:** Inspect incomplete data before designing preprocessing.

**Task:** Build `missing_only` with **only columns that have at least one missing value**, ordered from most to least missing.

<details><summary>Hint (open only if stuck)</summary><p>Start from df.isna().sum(); filter the Series &gt; 0; sort_values(ascending=False).</p></details>

In [ ]:
# TODO: find columns with any missing observations.
# missing_only = ...

In [ ]:
if "missing_only" in globals():
    assert (missing_only > 0).all()
    print("Practice 3: passed")
else:
    print("Practice 3: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 3</b></summary>
<pre><code>missing_only = df.isna().sum()
missing_only = missing_only[missing_only &gt; 0].sort_values(ascending=False)
print(missing_only)</code></pre>
</details>

## 4. EDA driven by a question: who survived?

**Why take an average here?** Because `Survived` is **0 or 1**, its mean is the **fraction of passengers who survived**. We are not averaging an arbitrary column; we are measuring an outcome rate across groups.

Hypotheses to test: does survival differ by **sex**, **ticket class**, or **age**?

**What to say in an interview:** “I checked target balance and compared the outcome across a few plausible features. These exploratory patterns help choose features, but correlation is not proof of causation.”

In [ ]:
print("Target counts (0=no, 1=yes):")
display(df["Survived"].value_counts().rename("passengers"))
print("Overall survival rate:", round(df["Survived"].mean(), 3))

by_sex = df.groupby("Sex")["Survived"].agg(survival_rate="mean", passengers="count")
by_class = df.groupby("Pclass")["Survived"].agg(survival_rate="mean", passengers="count")
print("\nSurvival by sex:")
display(by_sex)
print("\nSurvival by ticket class:")
display(by_class)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
by_sex["survival_rate"].plot.bar(ax=axes[0], title="Observed survival rate by sex")
by_class["survival_rate"].plot.bar(ax=axes[1], title="Observed survival rate by ticket class")
for ax in axes:
    ax.set_ylim(0, 1)
    ax.set_ylabel("Proportion survived")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### ✍️ Practice 4: EDA with a hypothesis

**Why this matters:** This is a realistic first-10-minutes interview task.

**Task:** Calculate `embarked_summary`, showing survival **rate** and **passenger count** for each departure port (`Embarked`). Then state which port has the highest observed rate.

<details><summary>Hint (open only if stuck)</summary><p>Group by Embarked; choose [&quot;Survived&quot;]; .agg(rate=&quot;mean&quot;, n=&quot;count&quot;); .sort_values(&quot;rate&quot;, ascending=False).</p></details>

In [ ]:
# TODO: make embarked_summary and display it.
# embarked_summary = ...

In [ ]:
if "embarked_summary" in globals():
    assert {"rate", "n"}.issubset(embarked_summary.columns)
    print("Practice 4: passed")
else:
    print("Practice 4: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 4</b></summary>
<pre><code>embarked_summary = (df.groupby(&quot;Embarked&quot;)[&quot;Survived&quot;]
                    .agg(rate=&quot;mean&quot;, n=&quot;count&quot;)
                    .sort_values(&quot;rate&quot;, ascending=False))
display(embarked_summary)</code></pre>
</details>

## 5. NumPy with a purpose: arrays, masks, and model outputs

**Pandas** is convenient for labeled tables; **NumPy** is convenient for numerical arrays and vectorized computations. scikit-learn often returns predictions and probabilities as NumPy arrays.

Here is a meaningful use case: calculate how many passengers we would label correctly if we predicted **everyone dies**. This is a *baseline* we want to beat later.

In [ ]:
labels = df["Survived"].to_numpy()   # Pandas Series -> NumPy array
always_zero = np.zeros(len(labels), dtype=int)
correct_mask = (always_zero == labels)  # vectorized comparisons return booleans
baseline_full_data_accuracy = correct_mask.mean()
print("labels:", labels[:10], "shape:", labels.shape)
print("correct mask:", correct_mask[:10])
print("Full-dataset majority-guess diagnostic:", round(baseline_full_data_accuracy, 3))
print("\nWhy NumPy matters: comparisons and .mean() operate over an entire array without Python loops.")

> **Important distinction:** the calculation above uses the full dataset **only to illustrate arrays and class balance**. We will use a majority baseline **based on training labels only** when evaluating on held-out data.

### ✍️ Practice 5: Vectorization in NumPy

**Why this matters:** ML metrics are typically array operations over predictions and labels.

**Task:** Create `survivors_mask` (a boolean array where true means survived) and `survivor_count_np` by summing it.

<details><summary>Hint (open only if stuck)</summary><p>Use == 1, and .sum() on the resulting boolean array.</p></details>

In [ ]:
# TODO: use the NumPy labels array defined above.
# survivors_mask = ...
# survivor_count_np = ...

In [ ]:
if "survivors_mask" in globals() and "survivor_count_np" in globals():
    assert survivor_count_np == int(df["Survived"].sum())
    print("Practice 5: passed")
else:
    print("Practice 5: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 5</b></summary>
<pre><code>survivors_mask = (labels == 1)
survivor_count_np = survivors_mask.sum()
print(survivor_count_np)</code></pre>
</details>

## 6. Define X/y and split the data — BEFORE fitting preprocessing

- **X (features):** information available to make a prediction.
- **y (target):** the correct answer, used to train and evaluate.
- **Training data:** used to learn model parameters and preprocessing statistics.
- **Validation data:** held out for measuring performance; do **not** fit imputers, scalers, encoders, or resamplers on it.

Our initial feature list is intentionally small and interpretable. We treat `Pclass` as categorical. We exclude IDs and high-cardinality text, but later we can engineer useful information from them if justified.

**Generic caution:** random stratified splits are appropriate for this independent-row tutorial; for temporal data use a time-based split, and for repeated users/entities use group-aware splits.

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]
feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["Survived"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("X train:", X_train.shape, "| X valid:", X_valid.shape)
print("Training survival rate:", round(y_train.mean(), 3))
print("Validation survival rate:", round(y_valid.mean(), 3))
assert "Survived" not in X_train.columns
assert set(X_train.index).isdisjoint(set(X_valid.index))

### ✍️ Practice 6: Train/validation split

**Why this matters:** This step is essential for trustworthy evaluation.

**Task:** Explain in one sentence why `df["Age"].median()` should NOT be used to fit an imputer before splitting. Then make `train_missing_age` equal to the number of missing ages in `X_train`.

<details><summary>Hint (open only if stuck)</summary><p>Use X_train[&quot;Age&quot;].isna().sum(). The imputation value must be learned only from training rows.</p></details>

In [ ]:
# TODO: add your explanation in a comment, then code below.
# train_missing_age = ...

In [ ]:
if "train_missing_age" in globals():
    assert int(train_missing_age) == int(X_train["Age"].isna().sum())
    print("Practice 6: passed")
else:
    print("Practice 6: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 6</b></summary>
<pre><code># Using the overall median leaks information about the validation distribution.
train_missing_age = X_train[&quot;Age&quot;].isna().sum()
print(train_missing_age)</code></pre>
</details>

## 7. A meaningful baseline

Before any fancy model, predict the most common class in the **training set** and measure accuracy on the **validation set**.

**Why:** a 65% accurate model can be useless if the majority class is already 90%. We need a reference point.

Other datasets require different baselines: median/mean for regression, last-known-value for forecasting, or simple rules for retrieval.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

majority_label = int(y_train.mode().iloc[0])
baseline_preds = np.full(shape=len(y_valid), fill_value=majority_label)
baseline_accuracy = accuracy_score(y_valid, baseline_preds)

print("Majority label from training data:", majority_label)
print("Validation baseline accuracy:", round(baseline_accuracy, 3))

## 8. Preprocessing: make raw CSV columns usable by a model

**Why?** `Age` contains missing values and `Sex` / `Embarked` contain text. A numerical model needs a consistent numeric feature matrix.

We'll build a **Pipeline** to learn preprocessing **only on each training split/fold**:

1. Numeric columns: fill missing values with **training median**, then scale for linear models.
2. Categorical columns: fill missing labels with **training mode**, then **one-hot encode** categories.
3. Combine both with `ColumnTransformer`.
4. Place the preprocessor and classifier in **one** pipeline — training and prediction use the same transformations.

**Why not `Sex.map()` separately in train and test?** It's easy to accidentally reverse encodings, which the uploaded notebook did. A shared fitted pipeline prevents this inconsistency.

**Why not SMOTE yet?** We have not established that it is needed. If we later resample, it must happen inside training folds — never before the split.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
print(preprocessor)

### Inspect what preprocessing does — without leaking

In this **demonstration only**, fit the transformer on training rows and transform a few validation rows. We will create the model pipeline next, so you will **not** need to manually transform data for fitting the classifier.

In [ ]:
from sklearn.base import clone

preprocessor_demo = clone(preprocessor)
transformed_train = preprocessor_demo.fit_transform(X_train)
transformed_valid = preprocessor_demo.transform(X_valid.iloc[:3])
print("Original columns:", len(feature_columns))
print("Transformed training matrix shape:", transformed_train.shape)
print("Transformed 3 validation rows shape:", transformed_valid.shape)
print("Sample transformed row:", np.asarray(transformed_valid[0]).ravel()[:12])

### ✍️ Practice 7: Choosing preprocessors

**Why this matters:** Modeling choices should follow feature semantics, not arbitrary patterns.

**Task:** If a new column `Country` held strings such as `US` and `IN`, would you add it to `numeric_features` or `categorical_features`? Write `country_feature_type = "..."` and explain why.

<details><summary>Hint (open only if stuck)</summary><p>Strings representing unordered categories require encoding, not numeric scaling.</p></details>

In [ ]:
# TODO: which list should the Country column go in?
# country_feature_type = "..."

In [ ]:
if "country_feature_type" in globals():
    assert country_feature_type == "categorical"
    print("Practice 7: passed")
else:
    print("Practice 7: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 7</b></summary>
<pre><code>country_feature_type = &quot;categorical&quot;
print(&quot;Use a fitted encoder for unordered country labels&quot;)</code></pre>
</details>

## 9. Train your first classifier — Logistic Regression

Despite its name, logistic regression is a **classification model**. It estimates a probability for class 1, then typically predicts class 1 if that probability is at least 0.5.

**Why first?** It is fast, gives a strong interpretable baseline, and usually requires fewer choices than more complex methods.

Notice that we pass **raw DataFrames** to the pipeline. The pipeline handles imputing, encoding, and scaling consistently.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

logreg.fit(X_train, y_train)
valid_pred_lr = logreg.predict(X_valid)
valid_proba_lr = logreg.predict_proba(X_valid)[:, 1]

print("First 10 predictions:", valid_pred_lr[:10])
print("First 10 probabilities of survival:", np.round(valid_proba_lr[:10], 3))
print("Validation accuracy:", round(accuracy_score(y_valid, valid_pred_lr), 3))

## 10. Evaluate: accuracy alone doesn't tell the whole story

- **Accuracy:** fraction of all predictions correct. Good when mistakes and class frequencies are balanced.
- **Precision (class 1):** of passengers predicted to survive, how many actually survived?
- **Recall (class 1):** of passengers who survived, how many did we identify?
- **F1:** balances precision and recall.
- **ROC-AUC:** measures how well probabilities rank survivors above nonsurvivors across thresholds.

**Transfer rule:** if the data is severely imbalanced (e.g., fraud), accuracy can be dangerously misleading. Choose metrics based on the cost of errors, not habit.

In [ ]:
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, confusion_matrix,
    roc_auc_score,
)

metrics_lr = {
    "accuracy": accuracy_score(y_valid, valid_pred_lr),
    "precision": precision_score(y_valid, valid_pred_lr, zero_division=0),
    "recall": recall_score(y_valid, valid_pred_lr, zero_division=0),
    "f1": f1_score(y_valid, valid_pred_lr, zero_division=0),
    "roc_auc": roc_auc_score(y_valid, valid_proba_lr),
}
display(pd.Series(metrics_lr, name="Logistic regression").round(3))
print(classification_report(y_valid, valid_pred_lr, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_valid, valid_pred_lr, display_labels=["Did not survive", "Survived"],
    cmap="Blues", colorbar=False
)
plt.title("Validation confusion matrix — Logistic Regression")
plt.tight_layout()
plt.show()

### Reading the confusion matrix

Rows = **actual** class; columns = **predicted** class (in the scikit-learn display above).

- Top-left: correct nonsurvivor predictions (**true negatives**).
- Top-right: predicted survival, but the person did not survive (**false positives**).
- Bottom-left: predicted no survival, but the person survived (**false negatives**).
- Bottom-right: correct survivor predictions (**true positives**).

Which mistake matters most depends on the application. For fraud detection, false negatives may be expensive; for some medical screenings, missing a case may be especially costly.

### ✍️ Practice 8: Calculate F1 and read errors

**Why this matters:** Roblox may ask you to justify the scoring metric, not merely fit a model.

**Task:** Compute `my_f1` for logistic-regression predictions and `false_negative_count` from `y_valid` and `valid_pred_lr`.

<details><summary>Hint (open only if stuck)</summary><p>f1_score(true, pred); false negatives are `(true == 1) &amp; (pred == 0)`. Count booleans with np.sum.</p></details>

In [ ]:
# TODO: two lines, one using sklearn and one using NumPy booleans.
# my_f1 = ...
# false_negative_count = ...

In [ ]:
if "my_f1" in globals() and "false_negative_count" in globals():
    assert np.isclose(my_f1, metrics_lr["f1"])
    assert false_negative_count >= 0
    print("Practice 8: passed")
else:
    print("Practice 8: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 8</b></summary>
<pre><code>my_f1 = f1_score(y_valid, valid_pred_lr)
false_negative_count = np.sum((y_valid.to_numpy() == 1) &amp; (valid_pred_lr == 0))
print(my_f1, false_negative_count)</code></pre>
</details>

## 11. Train a second model — Random Forest

A random forest averages the predictions of many decision trees. It can capture nonlinear relationships and interactions more naturally than logistic regression, but is less straightforward to interpret.

**Generic modeling principle:** compare a simple baseline with one reasonable alternative. Don't start by tuning 20 hyperparameters. Decide what to try from the **data and results**, not from a checklist.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=150, max_depth=6, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])
rf.fit(X_train, y_train)
valid_pred_rf = rf.predict(X_valid)
valid_proba_rf = rf.predict_proba(X_valid)[:, 1]

comparison = pd.DataFrame({
    "majority_baseline": {
        "accuracy": accuracy_score(y_valid, baseline_preds),
        "f1": f1_score(y_valid, baseline_preds, zero_division=0),
    },
    "logistic_regression": {
        "accuracy": accuracy_score(y_valid, valid_pred_lr),
        "f1": f1_score(y_valid, valid_pred_lr),
    },
    "random_forest": {
        "accuracy": accuracy_score(y_valid, valid_pred_rf),
        "f1": f1_score(y_valid, valid_pred_rf),
    }
}).T.round(3)
display(comparison)

**What does this result mean?** Compare against the majority baseline first. If model B scores higher than model A on this validation split, that is a useful signal — **not proof** B is universally better. We should avoid repeatedly tuning to the same validation set and later report cross-validation or a genuinely untouched test set.

**Original notebook comparison:** Your uploaded file used KNN and logistic regression, then tried SMOTE. We are teaching a simpler, leakage-safe baseline first. KNN/SMOTE are valid tools **when justified**, not mandatory steps.

### ✍️ Practice 9: Explain a modeling choice

**Why this matters:** Live interviews evaluate reasoning alongside code.

**Task:** Write 2–3 sentences in a comment: which model would you keep from the table above, which metric are you prioritizing, and what concern remains?

<details><summary>Hint (open only if stuck)</summary><p>Compare F1/accuracy, model complexity, and the fact that you have evaluated on one validation split.</p></details>

In [ ]:
# TODO: write your own answer in comments here.

<details><summary><b>Reveal reference answer for Practice 9</b></summary>
<pre><code># Example reasoning (do not memorize):
# I would choose based on the metric that matches the application&#x27;s error costs.
# I would favor the simpler model if scores were close, and validate stability across folds.</code></pre>
</details>

## 12. Feature engineering: create information the model can use

**Hypothesis:** traveling alone may relate to a passenger's outcome. We can derive family size from `SibSp` and `Parch`, which count companions of different types.

This is **row-wise deterministic feature engineering**: it uses only a passenger's own input fields, not target labels or dataset-wide statistics. It is safe to apply consistently to training and validation rows. Other feature engineering (e.g. target encoding) can leak labels and requires additional care.

We will compare with the previous model, while remembering that validation comparisons are exploratory.

In [ ]:
def add_family_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    return result

X_train_fe = add_family_features(X_train)
X_valid_fe = add_family_features(X_valid)

numeric_fe = numeric_features + ["FamilySize", "IsAlone"]
preprocess_fe = ColumnTransformer(transformers=[
    ("numeric", clone(numeric_pipeline), numeric_fe),
    ("categorical", clone(categorical_pipeline), categorical_features),
])
rf_fe = Pipeline(steps=[
    ("preprocess", preprocess_fe),
    ("model", RandomForestClassifier(
        n_estimators=150, max_depth=6, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])
rf_fe.fit(X_train_fe, y_train)
pred_rf_fe = rf_fe.predict(X_valid_fe)
print("RF original F1:", round(f1_score(y_valid, valid_pred_rf), 3))
print("RF with family features F1:", round(f1_score(y_valid, pred_rf_fe), 3))
print("Added features:", ["FamilySize", "IsAlone"])

### ✍️ Practice 10: Create a sensible feature

**Why this matters:** The useful question is "What signal might be missing?" rather than "How many columns can I create?"

**Task:** Create `train_fare_per_person` as `Fare / FamilySize` using `X_train_fe`. Think about when this ratio might be misleading.

<details><summary>Hint (open only if stuck)</summary><p>Pandas performs column-wise arithmetic; `FamilySize` is at least 1.</p></details>

In [ ]:
# TODO: use X_train_fe columns to compute a Series.
# train_fare_per_person = ...

In [ ]:
if "train_fare_per_person" in globals():
    assert len(train_fare_per_person) == len(X_train_fe)
    print("Practice 10: passed")
else:
    print("Practice 10: not attempted yet")

<details><summary><b>Reveal reference answer for Practice 10</b></summary>
<pre><code>train_fare_per_person = X_train_fe[&quot;Fare&quot;] / X_train_fe[&quot;FamilySize&quot;]
print(train_fare_per_person.head())</code></pre>
</details>

## 13. Cross-validation: is the score stable?

One train/validation split can be lucky or unlucky. **Stratified K-fold cross-validation** repeatedly creates training/validation folds with similar target balance. The **entire pipeline** refits on each training fold, which protects preprocessing from leakage.

**Important:** Here CV runs on the **original training partition**, leaving the final validation partition available. We are **not** claiming that cross-validation on a small dataset guarantees generalization. In a more rigorous project, use separate tuning and final test stages.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = cross_val_score(
    clone(logreg), X_train, y_train,
    cv=cv, scoring="accuracy", n_jobs=1
)
print("5-fold CV accuracies:", np.round(cv_acc, 3))
print(f"Mean: {cv_acc.mean():.3f} | Std: {cv_acc.std():.3f}")

## 14. Predict on new passengers, then export the model

The outcome for a new record is **unknown**; at prediction time we receive only feature values. This demonstrates why a pipeline is valuable: no manual mapping or independent imputer needs to be recreated.

These are hypothetical records; probabilities are outputs from an illustrative model, not statements about real individuals.

In [ ]:
new_passengers = pd.DataFrame([
    {"Age": 30, "Fare": 50.0, "SibSp": 0, "Parch": 0,
     "Pclass": 1, "Sex": "female", "Embarked": "S"},
    {"Age": np.nan, "Fare": 8.0, "SibSp": 1, "Parch": 0,
     "Pclass": 3, "Sex": "male", "Embarked": "Q"},
])

new_predictions = logreg.predict(new_passengers)
new_probabilities = logreg.predict_proba(new_passengers)[:, 1]
result = new_passengers.copy()
result["pred_survived"] = new_predictions
result["prob_survived"] = np.round(new_probabilities, 3)
display(result)

In [ ]:
import joblib

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
model_path = ARTIFACT_DIR / "titanic_logreg_pipeline.joblib"
joblib.dump(logreg, model_path)
reloaded_model = joblib.load(model_path)
assert np.array_equal(logreg.predict(new_passengers), reloaded_model.predict(new_passengers))
print(f"Saved and checked pipeline: {model_path}")

### Optional: official Kaggle competition submission

Only run this if you have downloaded the Titanic competition's **unlabeled** `test.csv` next to the notebook or in `data/`. The public 891-row training CSV is **not** the Kaggle test set.

After choosing your approach, retrain on **all labeled training data** to make competition predictions. This is appropriate for producing a Kaggle submission **after evaluation**; it does not provide a new independent accuracy estimate.

In [ ]:
kaggle_test_candidates = [Path("data/test.csv"), Path("test.csv")]
kaggle_test_path = next((p for p in kaggle_test_candidates if p.is_file()), None)

if kaggle_test_path is None:
    print("Optional submission skipped. Put official Kaggle test.csv in data/ or beside the notebook.")
else:
    test_df = pd.read_csv(kaggle_test_path)
    assert "PassengerId" in test_df.columns
    assert set(feature_columns).issubset(test_df.columns)
    final_model = clone(logreg).fit(X, y)
    kaggle_predictions = final_model.predict(test_df[feature_columns])
    submission = pd.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": kaggle_predictions.astype(int),
    })
    submission.to_csv(ARTIFACT_DIR / "submission.csv", index=False)
    print("Saved artifacts/submission.csv")
    display(submission.head())

## 15. Interview-ready reusable checklist

**Don't memorize Titanic numbers; memorize the reasoning workflow.**

**First 10 minutes:**
- Say what each row means, the target, task type, and what counts as success.
- `df.head()`, `df.shape`, `df.info()`, `df.isna().sum()`.
- `df[target].value_counts(normalize=True)` for classification.
- Spot IDs, timestamps, repeated entities, missingness, and possible leakage.

**Next 15 minutes:**
- Choose a split that matches data structure (stratified for independent classification rows; time-based for future prediction; grouped for repeated entities).
- Declare numeric and categorical feature lists.
- Build a `ColumnTransformer` and `Pipeline`; **learn preprocessing on training only**.

**Next 10 minutes:**
- Fit a simple model.
- Report an honest baseline and appropriate metrics.
- Run confusion matrix / error check.

**Last 10 minutes:**
- Try one justified alternative or feature; avoid rushed tuning.
- Explain limitations, possible leakage, imbalances, and next experiments.

### Quick syntax map

| Intent | Typical code |
|---|---|
| Load CSV | `pd.read_csv(path)` |
| Rows/columns | `df.shape` |
| Basic structure | `df.info()` |
| Missing counts | `df.isna().sum()` |
| Missing percent | `df.isna().mean() * 100` |
| Filter by condition | `df.loc[df['Age'] > 18, ['Age', 'Fare']]` |
| Group-level outcome rate | `df.groupby('Sex')['Survived'].mean()` |
| Make new column | `df['FamilySize'] = df['SibSp'] + df['Parch'] + 1` |
| NumPy array | `df['Survived'].to_numpy()` |
| Splitting | `train_test_split(X, y, stratify=y, random_state=42)` |
| Fit/predict | `model.fit(X_train, y_train)` / `model.predict(X_valid)` |
| Classification metric | `f1_score(y_valid, predictions)` |
| Export CSV | `df.to_csv('output.csv', index=False)` |

**Common traps:** leakage; mixing train/test encodings; using accuracy on imbalanced targets; dropping rows without checking how many; using columns that would not exist at prediction time; randomly splitting time series; fitting a scaler on the entire dataset; comparing models without a baseline.

## 16. Transfer task — 45-minute mock interview (no Titanic assumptions)

**Scenario:** You work on an online game platform. Each row is a player. Predict whether the player returns within 7 days (`retained_7d`, 0/1) using **activity from the first day only**. Imagine an unfamiliar CSV `players.csv` with columns like `user_id`, `country`, `device`, `sessions_day1`, `minutes_day1`, `first_purchase_day1`, and `retained_7d`.

Your **interview assignment** (use your own data or ask me for a CSV to practice on):

1. (0–5 min) Define the prediction target, granularity of rows, and what would constitute **target leakage**.
2. (5–15 min) Inspect schema, missingness, class balance, and 1–2 *motivated* segment comparisons.
3. (15–25 min) Choose a valid train/validation split and preprocessing pipeline.
4. (25–35 min) Compare a majority-class baseline with logistic regression (or random forest).
5. (35–45 min) Report F1, precision/recall, and an error breakdown. Suggest one next feature or experiment.

**Real-world caveat:** if each user appears many times, use a **user-grouped split**, and if you want to predict **future** retention, use a **time-based evaluation**. The simple stratified row split from Titanic would not automatically be correct.

### Self-scoring rubric /100

| Criterion | Points |
|---|---:|
| Defined problem, target, prediction-time inputs and leakage risks | 15 |
| Practical EDA and missingness inspection | 20 |
| Correct split and leakage-safe preprocessing | 25 |
| Functional model and meaningful baseline | 15 |
| Evaluation appropriate to the task | 15 |
| Clear explanation and next steps | 10 |

**Passing goal:** ≥80/100 with a working pipeline in 45 minutes, without copying the Titanic code.

## 17. Active-recall prompts (answer before your interview)

1. Why is `df['Age']` a Series and `df[['Age']]` a DataFrame?
2. Why is `df.groupby('Sex')['Survived'].mean()` a **survival rate** instead of a meaningless average?
3. Why should `Pclass` be treated as categorical even though the values are integers?
4. What information must never be placed in `X`? Give a Titanic example and a user-retention example.
5. Why split **before** fitting an imputer or scaler?
6. What does `OneHotEncoder(handle_unknown='ignore')` protect against?
7. Why might accuracy be a bad metric if only 1% of users churn?
8. What is the difference between `predict()` and `predict_proba()`?
9. If a tree model and logistic regression score similarly, how do you choose?
10. When is random stratified splitting inappropriate?

**Your finish line:** You can open a fresh CSV and construct the entire pipeline without being shown the exact function names. The data and features change; the reasoning pattern remains.

---
### Notes on this workbook and its source

This is a **new educational rewrite**, not a correction applied directly to your original Kaggle notebook. The original included KNN, logistic regression, a `ColumnTransformer`, SMOTE, and a Kaggle submission; this workbook keeps the **end-to-end problem**, modernizes preprocessing, and makes safe evaluation the core lesson.

**Dataset:** [Data Science Dojo Titanic CSV](https://github.com/datasciencedojo/datasets/blob/master/titanic.csv) (public copy of the Kaggle Titanic competition training data); [official Kaggle competition](https://www.kaggle.com/competitions/titanic/data).

**Reference docs:** [Pandas getting started](https://pandas.pydata.org/docs/getting_started/index.html), [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/compose.html), [common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html).